![imagenes](logo.png)

# Espacios normados y con producto interno
## Midiendo intensidad y parecido entre acordes

En el capítulo anterior construimos el universo infinito de combinaciones posibles de teclas: el espacio vectorial. Puedes sumar acordes, escalarlos (tocarlos más fuerte o más suave), pero aún nos falta responder preguntas esenciales:

- ¿Qué tan **fuerte** suena ese acorde en total? (intensidad, volumen, energía)
- ¿Qué tan **parecidos** son dos acordes? ¿Se refuerzan o se cancelan?
- ¿Cuánto "ángulo" hay entre dos melodías representadas como vectores?

Para eso dotamos al espacio de herramientas de **medición**: la **norma** (para medir tamaño/longitud) y el **producto interno** (para medir ángulos, similitud y proyecciones).

Y la joya que une todo: la **desigualdad de Cauchy-Schwarz**, que básicamente dice:  
**"el parecido entre dos cosas nunca puede superar el producto de sus intensidades individuales"**.

Es la base matemática de la similitud coseno, la estabilidad en regresión, la atención en transformers y casi todo lo que usamos en machine learning cuando hablamos de vectores.

In [ ]:
# Importamos las librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

# Configuración para gráficos más bonitos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Librerías cargadas correctamente")

## 1. La norma: ¿qué tan fuerte suena ese acorde?

La **norma** mide la "intensidad total" o "energía" de un vector. En nuestra analogía musical: si este acorde es una mezcla de notas, ¿cuánto volumen genera en conjunto?

### Definición formal

Una **norma** $\|\cdot\|$ es una función que asigna a cada vector un número ≥ 0 y cumple tres reglas:

| Propiedad | Fórmula | Analogía musical |
|-----------|---------|-------------------|
| No negatividad | $\|v\| \geq 0$ y $\|v\| = 0 \iff v = 0$ | No hay acorde que suene sin tener notas |
| Homogeneidad | $\|c v\| = |c| \|v\|$ | Duplicar la intensidad duplica el volumen |
| Desigualdad triangular | $\|u + v\| \leq \|u\| + \|v\|$ | Dos acordes juntos no suenan más fuerte que por separado |

### La norma más usada: Euclidiana (L²)

$$
\| \mathbf{v} \|_2 = \sqrt{v_1^2 + v_2^2 + \dots + v_n^2}
$$

In [ ]:
vector1 = np.array([1.0, 0.8, 1.2]) 
vector2 = np.array([0.9, 1.1, 0.0])   

norma_vec1 = np.linalg.norm(vector1)
norma_vec2 = np.linalg.norm(vector2)

In [ ]:
norma_vec1

In [ ]:
# Demostración de propiedades

#Homogeneidad: 
np.linalg.norm(2*vector1)

In [ ]:
# Desigualdad Triangular:
np.linalg.norm(vector1 + vector2) <= norma_vec1 + norma_vec2

**Preguntas**

1. Si triplicas todas las coordenadas de un vector, ¿qué pasa con su norma?
2. Si cambias el signo de todas las coordenadas de un vector, ¿cómo cambia la norma?
3. ¿Puede la norma de un vector ser negativa? ¿Por qué?

## 2. Producto interno: midiendo cuánto se "alinean" dos acordes

El **producto interno** (producto punto) captura similitud direccional y ángulo entre vectores.

En $\mathbb{R}^n$:

$$
\mathbf{u} \cdot \mathbf{v} = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n
$$

Y geométricamente:

$$
\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \, \|\mathbf{v}\| \cos \theta
$$

La **similitud coseno** normaliza para que el resultado esté entre $-1$ y $1$:

$$
\cos \theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}
$$

Esto es el corazón de **embeddings, búsqueda semántica, recomendaciones**, etc.

In [ ]:
# Ejemplo con embeddings ficticios de palabras
gato = np.array([1.0, 2.0, 3.0, 1.0])      # embedding de "gato"
felino = np.array([2.0, 4.0, 6.0, 2.0])    # embedding de "felino" (muy similar)
perro = np.array([1.0, 1.5, 2.5, 3.0])     # embedding de "perro" (algo similar)
coche = np.array([5.0, 0.0, 1.0, 8.0])     # embedding de "coche" (diferente)

def calcular_similitud(u, v, nombre_u="u", nombre_v="v"):
    dot = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    cos_sim = dot / (norm_u * norm_v)
    
    print(f"Producto interno {nombre_u}·{nombre_v}: {dot:.2f}")
    print(f"Similitud coseno: {cos_sim:.4f}")
    print(f"Ángulo (grados): {np.degrees(np.arccos(cos_sim)):.2f}°")
    return cos_sim

print("--- Comparación: gato vs felino ---")
sim_gato_felino = calcular_similitud(gato, felino, "gato", "felino")

print("\n--- Comparación: gato vs perro ---")
sim_gato_perro = calcular_similitud(gato, perro, "gato", "perro")

print("\n--- Comparación: gato vs coche ---")
sim_gato_coche = calcular_similitud(gato, coche, "gato", "coche")

### ⚠️ Confusión común

La similitud coseno mide **dirección**, no **magnitud**. Dos vectores [1,2,3] y [10,20,30] tienen coseno = 1 (perfectamente alineados) aunque sus normas sean muy diferentes.

En música: dos canciones con la misma "forma" pero una mucho más fuerte suenan "iguales" en términos de patrón, aunque una sea más ruidosa.

In [ ]:
# Demostración: vectores con misma dirección pero diferente magnitud
v1 = np.array([1, 2, 3])
v2 = 5 * v1  # [5, 10, 15]

dot = np.dot(v1, v2)
cos_sim = dot / (np.linalg.norm(v1) * np.linalg.norm(v2))

print(f"v1 = {v1}, ‖v1‖ = {np.linalg.norm(v1):.2f}")
print(f"v2 = {v2}, ‖v2‖ = {np.linalg.norm(v2):.2f}")
print(f"Similitud coseno: {cos_sim:.4f} (¡perfectamente alineados!)")

## 3. La desigualdad de Cauchy–Schwarz: el límite natural del parecido

**Desigualdad de Cauchy–Schwarz**

En cualquier espacio con producto interno:

$$
|\mathbf{u}\cdot\mathbf{v}| \le \|\mathbf{u}\|\,\|\mathbf{v}\|
$$

Con igualdad si y solo si $\mathbf{u}$ y $\mathbf{v}$ son paralelos (uno es múltiplo escalar del otro, o alguno es cero).

En palabras simples: **el parecido crudo nunca supera el producto de las intensidades**. No puedes tener más armonía de la que permiten las fuerzas individuales.

### Intuición geométrica

Del producto punto sabemos que:

$$
\mathbf{u}\cdot\mathbf{v} = \|\mathbf{u}\|\,\|\mathbf{v}\|\cos\theta
\quad \Rightarrow \quad
|\mathbf{u}\cdot\mathbf{v}| =
\|\mathbf{u}\|\,\|\mathbf{v}\|\,|\cos\theta|
\le
\|\mathbf{u}\|\,\|\mathbf{v}\|
$$

Porque $|\cos\theta|\le 1$ siempre.

In [ ]:
# Verificación experimental de Cauchy-Schwarz
np.random.seed(42)  # Para reproducibilidad

# Generamos 10 pares de vectores aleatorios
for i in range(10):
    u = np.random.randn(5)  # 5 dimensiones
    v = np.random.randn(5)
    
    producto = np.abs(np.dot(u, v))
    producto_normas = np.linalg.norm(u) * np.linalg.norm(v)
    
    print(f"Par {i+1}: |u·v| = {producto:.4f} ≤ {producto_normas:.4f} → {producto <= producto_normas}")

# Caso extremo: vectores paralelos
u = np.array([1, 2, 3])
v = 2 * u  # paralelo

producto = np.abs(np.dot(u, v))
producto_normas = np.linalg.norm(u) * np.linalg.norm(v)

print(f"\n--- Vectores paralelos ---")
print(f"|u·v| = {producto:.4f} = {producto_normas:.4f} (igualdad)")

### ✏️ Autoevaluación

¿Puede la similitud coseno entre dos vectores ser 1.5? ¿Por qué?

## 4. Ortogonalidad: acordes que no se interfieren

Dos vectores son **ortogonales** si:

$$
\mathbf{u}\cdot\mathbf{v} = 0
$$

(ángulo de $90^\circ$)

En datos: **variables ortogonales aportan información completamente independiente**, lo cual es ideal para modelos estables.

Ejemplo clásico: las **bases canónicas** $e_1, e_2, \dots, e_n$ son mutuamente ortogonales.

In [ ]:
# Bases canónicas en 3D
e1 = np.array([1, 0, 0])
e2 = np.array([0, 1, 0])
e3 = np.array([0, 0, 1])

print(f"e1·e2 = {np.dot(e1, e2)} → ortogonales")
print(f"e1·e3 = {np.dot(e1, e3)} → ortogonales")
print(f"e2·e3 = {np.dot(e2, e3)} → ortogonales")

# Visualización en 2D
plt.figure(figsize=(8, 8))
plt.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='red', label='e1 = [1,0]')
plt.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='blue', label='e2 = [0,1]')
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.grid(True)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.title('Vectores ortogonales (perpendiculares)')
plt.legend()
plt.gca().set_aspect('equal')
plt.show()

### ⚠️ Confusión común

La ortogonalidad (producto interno = 0) no significa "falta de relación" en el sentido coloquial. Significa que sus direcciones son perpendiculares. En datos: dos variables ortogonales son linealmente independientes, pero pueden tener relaciones no lineales.

## 5. Proyecciones: aislar la parte "relevante" de un acorde

La **proyección de** $\mathbf{u}$ **sobre** $\mathbf{v}$ es la componente de $\mathbf{u}$ en la dirección de $\mathbf{v}$:

$$
\operatorname{proj}_{\mathbf{v}} \mathbf{u}
=
\left(
\frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{v}\|^2}
\right)
\mathbf{v}
$$

En **ML**: se usa en **regresión (least squares)**, **PCA**, **residuales**, etc.

In [ ]:
def proyeccion(u, v):
    """Calcula la proyección de u sobre v"""
    factor = np.dot(u, v) / np.dot(v, v)
    return factor * v

# Ejemplo
u = np.array([3.0, 4.0, 5.0])
v = np.array([1.0, 0.0, 1.0])   # dirección de interés

proj = proyeccion(u, v)
perpendicular = u - proj  # componente ortogonal

print(f"Vector original u: {u}")
print(f"Dirección v: {v}")
print(f"Proyección de u sobre v: {proj}")
print(f"Componente perpendicular: {perpendicular}")

# Verificamos que son ortogonales
print(f"proj · perpendicular = {np.dot(proj, perpendicular):.10f} ≈ 0")

# Visualización en 2D simplificada
u_2d = np.array([3, 4])
v_2d = np.array([2, 1])
proj_2d = proyeccion(u_2d, v_2d)
perp_2d = u_2d - proj_2d

plt.figure(figsize=(8, 8))
plt.quiver(0, 0, u_2d[0], u_2d[1], angles='xy', scale_units='xy', scale=1, color='blue', label='u original')
plt.quiver(0, 0, v_2d[0], v_2d[1], angles='xy', scale_units='xy', scale=1, color='green', alpha=0.5, label='v dirección')
plt.quiver(0, 0, proj_2d[0], proj_2d[1], angles='xy', scale_units='xy', scale=1, color='red', label='proyección')
plt.quiver(proj_2d[0], proj_2d[1], perp_2d[0], perp_2d[1], angles='xy', scale_units='xy', scale=1, color='purple', label='componente perpendicular')
plt.xlim(-1, 5)
plt.ylim(-1, 5)
plt.grid(True)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.title('Proyección de u sobre v')
plt.legend()
plt.gca().set_aspect('equal')
plt.show()

## 6. Espacios métricos: el siguiente nivel

Hasta ahora hemos medido el **tamaño** de un vector (norma) y su **parecido/ángulo** (producto interno). Pero a veces lo que realmente nos interesa es: **¿qué tan lejos está un punto de otro?**

Un **espacio métrico** nos da una forma consistente de calcular distancias entre cualquier par de puntos.

### Definición formal

Un espacio métrico es un conjunto $V$ con una función **distancia** $d: V \times V \to \mathbb{R}$ que cumple:

| Propiedad | Fórmula | Analogía |
|-----------|---------|----------|
| No negatividad | $d(x,y) \geq 0$ | Las distancias no son negativas |
| Identidad | $d(x,y)=0 \iff x=y$ | Solo estás a distancia cero de ti mismo |
| Simetría | $d(x,y) = d(y,x)$ | La distancia de A a B es igual que de B a A |
| Desigualdad triangular | $d(x,z) \leq d(x,y) + d(y,z)$ | Ir directo es siempre el camino más corto |

## 7. Distancias más usadas en Machine Learning

### 7.1 Distancia Euclidiana (L²)

La clásica: "distancia en línea recta". Inducida por la norma L².

$$
d_{euclid}(\mathbf{x}, \mathbf{y}) = \|\mathbf{x} - \mathbf{y}\|_2 = \sqrt{\sum_{i=1}^n (x_i - y_i)^2}
$$

In [ ]:
# Ejemplo: distancia entre dos canciones representadas por características
cancion_A = np.array([8, 5, 2])  # [alegría, bailable, complejidad]
cancion_B = np.array([7, 9, 1])  # [alegría, bailable, complejidad]
cancion_C = np.array([2, 3, 9])  # canción muy diferente

dist_AB_euclid = np.linalg.norm(cancion_A - cancion_B)
dist_AC_euclid = np.linalg.norm(cancion_A - cancion_C)

print(f"Distancia Euclidiana A-B: {dist_AB_euclid:.3f}")
print(f"Distancia Euclidiana A-C: {dist_AC_euclid:.3f}")

### 7.2 Distancia Manhattan (L¹, "distancia del taxista")

Mide la distancia recorriendo ejes por separado. Útil cuando **los cambios en cada dimensión importan por separado** (ej. presupuestos, conteos, datos dispersos).

$$
d_{manh}(\mathbf{x},\mathbf{y}) = \sum_{i=1}^{n} |x_i - y_i|
$$

In [ ]:
dist_AB_manh = np.sum(np.abs(cancion_A - cancion_B))
dist_AC_manh = np.sum(np.abs(cancion_A - cancion_C))

print(f"Distancia Manhattan A-B: {dist_AB_manh}")
print(f"Distancia Manhattan A-C: {dist_AC_manh}")
print(f"\nObservación: Manhattan ≥ Euclidiana (|1|+| -4|+ |1| = 6 ≥ √18 ≈ 4.24)")

### 7.3 Distancia Coseno

Mide el ángulo entre vectores, ignorando su magnitud. Se define como:

$$
d_{cos}(\mathbf{x}, \mathbf{y}) = 1 - \cos(\theta) = 1 - \frac{\mathbf{x} \cdot \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}
$$

Va de 0 (vectores idénticos en dirección) a 2 (vectores opuestos).

In [ ]:
def distancia_coseno(u, v):
    """Calcula la distancia coseno entre dos vectores"""
    cos_sim = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
    return 1 - cos_sim

# Ejemplo con vectores que tienen diferente magnitud pero misma dirección
v1 = np.array([1, 2, 3])
v2 = 10 * v1  # misma dirección, magnitud 10 veces mayor
v3 = np.array([-1, -2, -3])  # dirección opuesta
v4 = np.array([3, 1, 2])  # dirección diferente

print("--- Distancia coseno ---")
print(f"v1 y v2 (misma dirección): {distancia_coseno(v1, v2):.4f} (cercanía máxima)")
print(f"v1 y v3 (opuestos): {distancia_coseno(v1, v3):.4f} (máxima distancia)")
print(f"v1 y v4 (diferentes): {distancia_coseno(v1, v4):.4f}")

### Comparación visual de distancias

In [ ]:
# Visualización de las "bolas unitarias" para diferentes métricas en 2D
theta = np.linspace(0, 2*np.pi, 100)

# Euclidiana: círculo
x_euclid = np.cos(theta)
y_euclid = np.sin(theta)

# Manhattan: rombo
# Para Manhattan, |x| + |y| = 1
t = np.linspace(-1, 1, 50)
x_manh = np.concatenate([t, np.ones_like(t), -t, -np.ones_like(t)])
y_manh = np.concatenate([1 - np.abs(t), -t, -1 + np.abs(t), t])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_euclid, y_euclid, 'b-', linewidth=2, label='Euclidiana (círculo)')
plt.plot(x_manh, y_manh, 'r-', linewidth=2, label='Manhattan (rombo)')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.grid(True, alpha=0.3)
plt.title('Bola unitaria: puntos a distancia 1 del origen')
plt.legend()
plt.axis('equal')
plt.xlim(-1.5, 1.5)
plt.ylim(-1.5, 1.5)

plt.subplot(1, 2, 2)
# Para coseno no hay "bola unitaria" porque no es una métrica inducida por norma
# pero podemos visualizar puntos con misma distancia coseno desde [1,0]
puntos = np.random.randn(100, 2)
puntos = puntos / np.linalg.norm(puntos, axis=1, keepdims=True)  # Normalizar a norma 1
origen = np.array([1, 0])

distancias_cos = [distancia_coseno(origen, p) for p in puntos]
colores = plt.cm.viridis(distancias_cos / max(distancias_cos))

plt.scatter(puntos[:, 0], puntos[:, 1], c=colores, alpha=0.7)
plt.scatter(origen[0], origen[1], c='red', s=200, marker='*', label='Referencia [1,0]')
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.grid(True, alpha=0.3)
plt.title('Distancia coseno desde [1,0] (color más oscuro = más cerca)')
plt.legend()
plt.axis('equal')
plt.xlim(-1.5, 1.5)
plt.ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## 8. ¿Por qué son importantes las métricas en Machine Learning?

Las métricas están en el corazón de casi todos los algoritmos de ML:

| Algoritmo | Métrica típica | ¿Para qué la usa? |
|-----------|----------------|-------------------|
| **K-Nearest Neighbors (KNN)** | Euclidiana, Manhattan | Encontrar los k puntos más cercanos |
| **K-Means** | Euclidiana | Asignar puntos al centroide más cercano |
| **DBSCAN** | Euclidiana | Identificar regiones densas |
| **Recomendadores** | Coseno | Encontrar ítems o usuarios similares |
| **PCA** | Euclidiana (implícita) | Maximizar varianza (distancias al cuadrado) |
| **Regresión** | Euclidiana al cuadrado | Minimizar error cuadrático |
| **Árboles de decisión** | No necesitan métrica | Divisiones por características |

### Ejemplo práctico: ¿Qué métrica elegir para recomendar canciones?

Imaginemos que representamos canciones con dos características:
- **Tempo**: lento (0) a rápido (10)
- **Alegría**: triste (0) a feliz (10)

In [ ]:
# Dataset ficticio de canciones
canciones = pd.DataFrame({
    'canción': ['Bohemian Rhapsody', 'Despacito', 'Happy', 'Claire de Lune', 'Smells Like Teen Spirit'],
    'tempo': [7, 8, 9, 2, 6],
    'alegría': [5, 7, 10, 3, 3],
    'artista': ['Queen', 'Luis Fonsi', 'Pharrell', 'Debussy', 'Nirvana']
})

print("🎵 Dataset de canciones:")
print(canciones)

# Cancion de referencia
ref = canciones.iloc[0][['tempo', 'alegría']].values.astype(float)

print("\n--- Comparando con 'Bohemian Rhapsody' ---")
for idx, row in canciones.iterrows():
    vec = row[['tempo', 'alegría']].values.astype(float)
    
    dist_euc = np.linalg.norm(ref - vec)
    dist_man = np.sum(np.abs(ref - vec))
    dist_cos = distancia_coseno(ref, vec)
    
    print(f"\n{row['canción']}:")
    print(f"  Euclidiana: {dist_euc:.3f}")
    print(f"  Manhattan:  {dist_man:.3f}")
    print(f"  Coseno:     {dist_cos:.3f}")

### Discusión: ¿Qué métrica elegirías?

- **Euclidiana**: Si te importa la cercanía global (tempo Y alegría deben ser parecidos)
- **Manhattan**: Si cada característica contribuye independientemente (útil si son independientes)
- **Coseno**: Si te importa el *perfil* (proporción tempo/alegría) más que la magnitud. Ideal para recomendaciones donde algunas canciones son más "intensas" en todo.

**¿Qué pasa con 'Claire de Lune'?**  
Es muy diferente en Euclidiana/Manhattan (valores bajos), pero su perfil (tempo=2, alegría=3) tiene proporción similar a 'Smells Like Teen Spirit' (tempo=6, alegría=3) → en coseno podrían ser más cercanas de lo que esperarías.

